# GAN-based AI Text Detection
Détection de texte généré par IA via un GAN exploitant les embeddings BERT.

## 1. Installation des dépendances

In [ ]:
!pip install transformers scikit-learn -q

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import random
import string

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertEncoder
from sklearn.metrics import roc_auc_score

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

## 3. Chargement des données

**Option A (recommandée) — Upload manuel :**  
Exécutez la cellule suivante et uploadez les 4 fichiers CSV depuis le dossier `llm-detect-ai-generated-text`.

**Option B — Google Drive :** commentez le bloc upload et décommentez le bloc Drive.

In [ ]:
# --- Option A : upload direct ---
from google.colab import files
uploaded = files.upload()  # Sélectionnez train_essays.csv, test_essays.csv, train_prompts.csv, sample_submission.csv

TRAIN_PATH  = "train_essays.csv"
TEST_PATH   = "test_essays.csv"
PROMPT_PATH = "train_prompts.csv"

# --- Option B : Google Drive (décommentez si préféré) ---
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/llm-detect-ai-generated-text/"
# TRAIN_PATH  = BASE + "train_essays.csv"
# TEST_PATH   = BASE + "test_essays.csv"
# PROMPT_PATH = BASE + "train_prompts.csv"

src_train  = pd.read_csv(TRAIN_PATH)
src_prompt = pd.read_csv(PROMPT_PATH)
src_sub    = pd.read_csv(TEST_PATH)

# Mélange du jeu d'entraînement pour une répartition équitable
src_train = src_train.sample(frac=1, random_state=42).reset_index(drop=True)

print("Train shape :", src_train.shape)
print("Test  shape :", src_sub.shape)
print("Distribution des labels :\n", src_train['generated'].value_counts())
src_train.head()

## 4. Préparation du modèle BERT

In [ ]:
tokenizer_save_path = "/content/bert_tokenizer"
model_save_path     = "/content/bert_model"

# Chargement depuis HuggingFace Hub
tokenizer       = BertTokenizer.from_pretrained('bert-base-uncased')
pretrained_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
pretrained_model.to(device)
pretrained_model.eval()

# On extrait uniquement l'encodeur BERT (sans la tête de classification)
# et on gèle ses poids : il sert uniquement à produire des embeddings
embedding_model = pretrained_model.bert
for param in embedding_model.parameters():
    param.requires_grad = False

print("Modèle BERT chargé et paramètres gelés.")

## 5. Définition des hyperparamètres

In [ ]:
train_batch_size = 8      # Petit batch pour tenir en mémoire GPU
test_batch_size  = 16
lr               = 2e-4   # Learning rate standard pour les GANs
beta1            = 0.5    # Beta1 Adam recommandé pour les GANs
nz               = 100    # Dimensions du vecteur latent
num_epochs       = 5
num_hidden_layers = 3     # Couches du BertEncoder du Générateur (plus léger)
train_ratio      = 0.8

## 6. Préparation des données

In [ ]:
class GANDAIGDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


all_num   = len(src_train)               # Nombre total d'exemples
train_num = int(all_num * train_ratio)   # Exemples d'entraînement
test_num  = all_num - train_num          # Exemples de validation

train_set = src_train[:train_num]
test_set  = pd.concat([
    src_train[train_num:],               # Portion de validation tirée du train
]).reset_index(drop=True)

# Les labels sont convertis en FloatTensor pour BCELoss
train_dataset = GANDAIGDataset(
    train_set['text'].tolist(),
    torch.FloatTensor(train_set['generated'].tolist())
)
test_dataset = GANDAIGDataset(
    test_set['text'].tolist(),
    torch.FloatTensor(test_set['generated'].tolist())
)

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size,  shuffle=False)

print(f"Train : {train_num} exemples | Validation : {test_num} exemples")
print(f"Batches train : {len(train_loader)} | Batches val : {len(test_loader)}")

## 7. Définition du Générateur

Architecture :  
`bruit (nz)` → `Linear` → `reshape (256, 128)` → `ConvTranspose1d × 2` → `(768, 128)` → `permute` → `(128, 768)` → `BertEncoder` → embeddings de forme `(batch, seq=128, hidden=768)`

In [ ]:
# Configuration partagée entre Générateur et Discriminateur
# hidden_size=768 par défaut (compatible BERT base)
config = BertConfig(num_hidden_layers=num_hidden_layers)


class Generator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Linear(input_dim, 256 * 128)

        self.conv_net = nn.Sequential(
            # (batch, 256, 128) → (batch, 512, 128)  [stride=1, padding=1]
            nn.ConvTranspose1d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            # (batch, 512, 128) → (batch, 768, 128)  [stride=1, padding=1]
            nn.ConvTranspose1d(512, 768, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(768),
            nn.Tanh(),
        )
        # BertEncoder aléatoire (poids initialisés aléatoirement, appris pendant le GAN)
        self.bert_encoder = BertEncoder(config)

    def forward(self, x):
        x = self.fc(x)                  # (batch, 256*128)
        x = x.view(x.size(0), 256, 128) # (batch, 256, 128) — channels × longueur
        x = self.conv_net(x)            # (batch, 768, 128)
        x = x.permute(0, 2, 1)          # (batch, 128, 768) — format BertEncoder
        x = self.bert_encoder(x)        # BaseModelOutput avec .last_hidden_state
        return x

## 8. Définition du Discriminateur

Architecture :  
Embeddings `(batch, seq, 768)` → `BertEncoder (6 couches pré-entraînées)` → `SumBertPooler` → `(batch, 768)` → `Linear → ReLU → Dropout → Linear` → `sigmoid` → probabilité `[0, 1]`

Convention des labels : **0 = texte humain**, **1 = texte généré par IA**

In [ ]:
class SumBertPooler(torch.nn.Module):
    """Pooling par somme normalisée sur la dimension séquence."""
    def __init__(self):
        super().__init__()

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        sum_hidden = hidden_states.sum(dim=1)          # (batch, 768)
        sum_mask   = sum_hidden.sum(1).unsqueeze(1)    # (batch, 1)
        sum_mask   = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_hidden / sum_mask        # normalisation
        return mean_embeddings


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # On crée un BertEncoder puis on remplace ses couches par les 6 premières
        # couches du modèle pré-entraîné (poids BERT réels)
        self.bert_encoder = BertEncoder(config)
        self.bert_encoder.layer = nn.ModuleList([
            layer for layer in pretrained_model.bert.encoder.layer[:6]
        ])
        self.pooler = SumBertPooler()
        self.classifier = torch.nn.Sequential(
            nn.Linear(768, 384),  # (batch, 768) → (batch, 384)
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(384, 1),    # (batch, 384) → (batch, 1)
        )

    def forward(self, input):
        out = self.bert_encoder(input)          # BaseModelOutput
        out = self.pooler(out.last_hidden_state) # (batch, 768)
        out = self.classifier(out)              # (batch, 1)
        return torch.sigmoid(out).view(-1)      # (batch,) ∈ [0, 1]

## 9. Fonctions utilitaires d'entraînement

In [ ]:
def eval_auc(model):
    """Évalue le discriminateur avec l'AUC-ROC sur le jeu de validation."""
    model.eval()
    predictions = []
    actuals     = []

    with torch.no_grad():
        for batch in test_loader:
            encodings      = tokenizer(
                list(batch[0]), padding=True, truncation=True, return_tensors="pt"
            ).to(device)
            input_ids      = encodings['input_ids']
            token_type_ids = encodings['token_type_ids']
            attention_mask = encodings['attention_mask']  # disponible si nécessaire

            embeded = embedding_model(
                input_ids=input_ids, token_type_ids=token_type_ids
            )
            embeded = embeded.last_hidden_state  # (batch, seq, 768)

            label = batch[1].float().to(device)

            outputs = model(embeded)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(label.cpu().numpy())

    try:
        auc = roc_auc_score(actuals, predictions)
    except ValueError:
        # Peut arriver si un seul label est présent dans le batch de validation
        auc = 0.5
    print("AUC:", auc)
    return auc


def get_model_info_dict(model, epoch, auc_score):
    """Sauvegarde les poids du modèle et son score pour sélectionner le meilleur."""
    current_device = next(model.parameters()).device
    model.to('cpu')
    model_info = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'auc_score': auc_score,
    }
    model.to(current_device)
    return model_info


def preparation_embedding(texts):
    """
    Tokenise les textes et retourne les embeddings BERT (last_hidden_state).
    Retourne un tenseur (batch, seq_len, 768) sur `device`.
    """
    encodings = tokenizer(
        list(texts), padding=True, truncation=True, return_tensors="pt"
    )
    input_ids      = encodings['input_ids'].to(device)
    token_type_ids = encodings['token_type_ids'].to(device)
    embeded = embedding_model(
        input_ids=input_ids, token_type_ids=token_type_ids
    )
    return embeded.last_hidden_state  # tenseur (batch, seq_len, 768)


def GAN_step(optimizerG, optimizerD, netG, netD, real_data, label, epoch, i):
    """
    Une étape d'entraînement GAN :
      - Discriminateur : apprend à classer les vrais textes (labels 0/1)
                         ET à classer les faux comme IA (label=1).
      - Générateur     : apprend à produire des embeddings classés comme humains (label=0).
    """
    # --- Entraînement du Discriminateur ---
    netD.zero_grad()
    batch_size = real_data.size(0)

    # Passe sur les vrais données (labels réels : 0=humain, 1=IA)
    output      = netD(real_data)
    errD_real   = criterion(output, label)
    errD_real.backward()
    D_x = output.mean().item()

    # Génération de faux embeddings et passe sur les faux (toujours labelisés 1 = IA)
    noise     = torch.randn(batch_size, nz, device=device)
    fake_data = netG(noise).last_hidden_state
    label.fill_(1)  # faux textes = IA par convention
    output      = netD(fake_data.detach())
    errD_fake   = criterion(output, label)
    errD_fake.backward()
    D_G_z1 = output.mean().item()
    errD   = errD_real + errD_fake
    optimizerD.step()

    # --- Entraînement du Générateur ---
    netG.zero_grad()
    label.fill_(0)  # le générateur veut être perçu comme humain (0)
    output = netD(fake_data)
    errG   = criterion(output, label)
    errG.backward()
    D_G_z2 = output.mean().item()
    optimizerG.step()

    if i % 50 == 0:
        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, num_epochs, i, len(train_loader),
                 errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

    return optimizerG, optimizerD, netG, netD

## 10. Entraînement

In [ ]:
netG = Generator(nz).to(device)
netD = Discriminator().to(device)

criterion  = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

model_infos = []

for epoch in range(num_epochs):
    netG.train()
    netD.train()

    for i, data in enumerate(train_loader, 0):
        # Embeddings calculés sans gradient (BERT gelé)
        with torch.no_grad():
            embeded = preparation_embedding(data[0])  # (batch, seq, 768) sur device

        optimizerG, optimizerD, netG, netD = GAN_step(
            optimizerG=optimizerG,
            optimizerD=optimizerD,
            netG=netG,
            netD=netD,
            real_data=embeded.to(device),
            label=data[1].float().to(device),
            epoch=epoch,
            i=i,
        )

    # Évaluation AUC à la fin de chaque époque
    auc_score = eval_auc(netD)
    model_infos.append(get_model_info_dict(netD, epoch, auc_score))

print('Entraînement terminé !')

## 11. Inférence

On charge le discriminateur ayant obtenu le meilleur AUC sur la validation.

In [ ]:
# Sélection du meilleur modèle selon l'AUC
max_auc_model_info = max(model_infos, key=lambda x: x['auc_score'])
print(f"Meilleur AUC : {max_auc_model_info['auc_score']:.4f} (époque {max_auc_model_info['epoch']})")

model = Discriminator()
model.load_state_dict(max_auc_model_info['model_state_dict'])
model.to(device)
model.eval()


class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __getitem__(self, idx):
        return self.texts[idx]

    def __len__(self):
        return len(self.texts)


sub_dataset      = InferenceDataset(src_sub['text'].tolist())
inference_loader = DataLoader(sub_dataset, batch_size=test_batch_size, shuffle=False)

sub_predictions = []
with torch.no_grad():
    for batch in inference_loader:
        encodings = tokenizer(
            list(batch), padding=True, truncation=True, return_tensors="pt"
        ).to(device)
        input_ids      = encodings['input_ids']
        token_type_ids = encodings['token_type_ids']

        embeded = embedding_model(
            input_ids=input_ids, token_type_ids=token_type_ids
        )
        embeded = embeded.last_hidden_state.to(device)  # (batch, seq, 768)

        outputs = model(embeded)
        sub_predictions.extend(outputs.cpu().numpy())

# Fichier de soumission
sub_ans_df = pd.DataFrame({
    'id':        src_sub['id'].values,
    'generated': sub_predictions,
})
print(sub_ans_df)
sub_ans_df.to_csv("submission.csv", index=False)
print("\nFichier submission.csv sauvegardé.")